# Causal Discovery Analysis

Causal search using BOSS/FGES algorithms via PyTetrad.

**Reference:** Kumu R package, `issue_causal_analysis.Rmd` 

## Notebook Setup Instructions

This guide is for code reviewers to run the notebook end-to-end without additional context.

### Environment Requirements

- **OS:** macOS/Linux/Windows (tested in this repo on macOS).
- **Python:** 3.10+ recommended (repo requirement is `>=3.7`; avoid very old Python).
- **Java:** JDK 11+ recommended (JDK 17 works well for Tetrad-based workflows).
- **Jupyter:** VS Code Notebook or Jupyter Lab.

### Required Python Packages

Install from the repository root (`pykumu/`) in a clean virtual environment:

```bash
python3 -m venv .venv
source .venv/bin/activate  # Windows: .venv\\Scripts\\activate
python -m pip install --upgrade pip
pip install numpy pandas JPype1
```

If `pytetrad` is not already importable, install local package mode from this repo root:

```bash
pip install -e .
```

### Required Files (in `causal_analysis_notebook/`)

- `null_variable_dt.csv` — default input dataset (raw or preprocessed format both supported)
- `mike_knowledge_box.txt` — domain-knowledge constraints

### Working Directory and Kernel

- Open notebook: `causal_analysis_notebook/causal_analysis_notebook.ipynb`.
- Ensure working directory resolves notebook-relative files (`causal_analysis_notebook/`).
- Select the same Python interpreter where dependencies were installed.
- If helper modules change, **restart kernel** before re-running.

### Configuration Checklist (Cell 4)

- `ALGORITHM`: `"boss"` or `"fges"`
- `DATA_PATH`: path to the input CSV (default: `null_variable_dt.csv`); swap to any other file without changing anything else
- `PROCESSED_DATA_PATH`: output path for the null-variable dataset (never overwrites the input)
- `KNOWLEDGE_FILE`: verify path exists (default: `mike_knowledge_box.txt`)
- `N_BOOTSTRAP`: lower this for quick validation runs, increase for full analysis

### Recommended Run Order

1. Run **Configuration** and **Import Libraries**.
2. Run all **Feature Engineering** cells top-to-bottom.
3. Run **FGES Null Variable Search** (produces null-search graph output).
4. Run **Deriving the 1 PNEF Threshold** (`pnef_1` must be created).
5. Run **Non-Null Causal Search**.
6. Run **Applying 1PNEF Threshold**.
7. Run **Results** sections (full graph, subgraph, cycle detection).

### Troubleshooting

- **`AttributeError` on Tetrad methods:** restart kernel and rerun from top (module reload issue).
- **Java heap/memory issues:** reduce `N_BOOTSTRAP`, keep `SHOW_BOOTSTRAP_OUTPUT = False`, rerun.
- **File not found:** verify you are in `causal_analysis_notebook/` and required CSV/knowledge files exist.
- **Import errors (`pytetrad`, `jpype`):** confirm the active interpreter matches the environment where packages were installed.

## Configuration

Set algorithm parameters and file paths.

In [1]:
# Analysis Parameters
ALGORITHM = "boss"  # Options: "boss" or "fges"

# Input/output paths
DATA_PATH = "null_variable_dt.csv"           # Input CSV (raw or preprocessed format)
PROCESSED_DATA_PATH = "null_variable_processed_dt.csv"  # Output: null-variable dataset
BINARIZED_DATA_PATH = "binarized_variable_dt.csv"       # Output: dataset without null variables

KNOWLEDGE_FILE = "mike_knowledge_box.txt"
OUTPUT_DIR = "boss_results" if ALGORITHM == "boss" else "fges_results"
NON_NULL_OUTPUT_DIR = "boss_domain_results" if ALGORITHM == "boss" else "fges_domain_results"

# Algorithm-specific parameters
N_BOOTSTRAP = 80 if ALGORITHM == "boss" else 50

# Output control
SHOW_BOOTSTRAP_OUTPUT = False  # Set to True to see bootstrap iteration counts

## Import Libraries

Load required modules for data processing and causal analysis.

In [2]:
# Increase Java memory allocation for large bootstrap analyses
import os
os.environ['JAVA_TOOL_OPTIONS'] = '-Xmx8g -Xms4g'  # 8GB max heap, 4GB initial
print("✓ Java memory configured: 8GB max, 4GB initial")

✓ Java memory configured: 8GB max, 4GB initial


In [3]:
import sys
import os

# Ensure the local pytetrad (which has get_json) takes precedence over any
# system-installed version. The repo root is one level above this notebook.
_repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pandas as pd
import numpy as np
import json
import time
import pytetrad.tools.TetradSearch as ts

print(f"pytetrad loaded from: {ts.__file__}")


def parse_graph(graph_json_str):
    """
    Parse Tetrad JSON graph into nodes, edgeset, and edge_type_probabilities.

    Matches R function: parse_graph() from kumu.

    The edgeset table contains the ensemble edge for each node pair. Because we
    are performing multiple executions (bootstrap), the probabilities represent
    the ensemble of all edges formed on each execution.
    """
    graph_data = json.loads(str(graph_json_str))

    nodes = pd.DataFrame({
        'node_name': [node['name'] for node in graph_data.get('nodes', [])]
    })

    edgeset_rows = []
    edge_type_prob_rows = []

    for edge in graph_data.get('edgesSet', []):
        node1_name  = edge['node1']['name']
        node2_name  = edge['node2']['name']
        endpoint1   = edge.get('endpoint1', '')
        endpoint2   = edge.get('endpoint2', '')
        bold        = edge.get('bold', False)
        highlighted = edge.get('highlighted', False)
        properties  = ';'.join(edge.get('properties', []))
        probability = edge.get('probability', 1.0)

        edgeset_rows.append({
            'node1_name':  node1_name,
            'node2_name':  node2_name,
            'endpoint1':   endpoint1,
            'endpoint2':   endpoint2,
            'bold':        bold,
            'highlighted': highlighted,
            'properties':  properties if properties else None,
            'probability': probability
        })

        for etp in edge.get('edgeTypeProbabilities', []):
            edge_type_prob_rows.append({
                'node1_name': node1_name,
                'node2_name': node2_name,
                'edge_type':  etp.get('edgeType', ''),
                'properties': ';'.join(etp.get('properties', [])) or None,
                'probability': etp.get('probability', 0.0)
            })

    edgeset = pd.DataFrame(edgeset_rows) if edgeset_rows else pd.DataFrame(
        columns=['node1_name', 'node2_name', 'endpoint1', 'endpoint2',
                 'bold', 'highlighted', 'properties', 'probability'])

    edge_type_probabilities = pd.DataFrame(edge_type_prob_rows) if edge_type_prob_rows else pd.DataFrame(
        columns=['node1_name', 'node2_name', 'edge_type', 'properties', 'probability'])

    return {
        'nodes': nodes,
        'edgeset': edgeset,
        'edge_type_probabilities': edge_type_probabilities
    }

pytetrad loaded from: /Users/phuonghuupham/pykumu/pytetrad/tools/TetradSearch.py


Picked up JAVA_TOOL_OPTIONS: -Xmx8g -Xms4g


# Feature Engineering


## Formatting Data Types

In order to be loaded in Tetrad, some variables must be transformed from String to Integer due to data type limitations. 

### CVE Data Type

We concatenate the last two digits of the year with the last four digits of the cve_id and convert into an integer. (E.g. 2006 and CVE ID XXX4339 becomes 06339).


In [4]:
raw_dt = pd.read_csv(DATA_PATH)
print(f"Loaded: {raw_dt.shape[0]} rows × {raw_dt.shape[1]} columns")

# CVE Data Type: encode cve_id if present; otherwise reconstruct from b_ indicator columns
if "cve_id" in raw_dt.columns:
    cve_as_str = raw_dt["cve_id"].astype(str)
    last_two_digits_year = cve_as_str.str.slice(6, 8)
    last_four_digits_cve = cve_as_str.str.slice(-4)
    raw_dt["cve_id"] = pd.to_numeric(last_two_digits_year + last_four_digits_cve, errors="coerce")
else:
    b_cols_present = [c for c in raw_dt.columns if c.startswith("b_")]
    if b_cols_present:
        b_mat = raw_dt[b_cols_present].to_numpy()
        b_idx = b_mat.argmax(axis=1)
        has_signal = b_mat.sum(axis=1) > 0
        raw_dt["cve_id"] = [
            int(b_cols_present[b_idx[r]].replace("b_", "")) if has_signal[r] else r
            for r in range(len(raw_dt))
        ]
    else:
        raw_dt["cve_id"] = range(len(raw_dt))

# Activity features: derive from commit_interval if present, else from commit counts
if "commit_interval" in raw_dt.columns:
    commit_interval = raw_dt["commit_interval"].fillna("").astype(str)
    raw_dt["activity_0"] = np.where(commit_interval.eq(""), 1, 0)
    raw_dt["activity_2"] = np.where(commit_interval.ne(""), 1, 0)
elif "commit" in raw_dt.columns:
    raw_dt["activity_0"] = np.where(raw_dt["commit"] <= 0, 1, 0)
    raw_dt["activity_2"] = np.where(raw_dt["commit"] > 0, 1, 0)
else:
    raw_dt["activity_0"] = 0
    raw_dt["activity_2"] = 0

print("Completed: CVE Data Type")

Loaded: 4870 rows × 138 columns
Completed: CVE Data Type


/var/folders/qg/19_8zd9x78b4fcgq8j_d8jmc0000gn/T/ipykernel_12215/3590379811.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw_dt["cve_id"] = [
/var/folders/qg/19_8zd9x78b4fcgq8j_d8jmc0000gn/T/ipykernel_12215/3590379811.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw_dt["activity_0"] = np.where(raw_dt["commit"] <= 0, 1, 0)
/var/folders/qg/19_8zd9x78b4fcgq8j_d8jmc0000gn/T/ipykernel_12215/3590379811.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

### Convert "start" to Unix Timestamp

To use start in causal analysis, we convert it to a unix timestamp. 

In [5]:
# Convert "start" to Unix Timestamp
# If already numeric (unix timestamp), leave it as-is; otherwise parse from string/datetime
if "start_datetime" in raw_dt.columns:
    raw_dt["start"] = pd.to_datetime(raw_dt["start_datetime"], errors="coerce", utc=True)
    raw_dt["start"] = raw_dt["start"].map(lambda x: x.timestamp() if pd.notna(x) else np.nan)
elif "start" in raw_dt.columns:
    if pd.api.types.is_numeric_dtype(raw_dt["start"]):
        pass  # already a Unix timestamp
    else:
        raw_dt["start"] = pd.to_datetime(raw_dt["start"], errors="coerce", utc=True)
        raw_dt["start"] = raw_dt["start"].map(lambda x: x.timestamp() if pd.notna(x) else np.nan)
else:
    raise ValueError("Expected either 'start_datetime' or 'start' column in data")

print("Completed: Convert 'start' to Unix Timestamp")

Completed: Convert 'start' to Unix Timestamp


## Feature Renaming

In [6]:
# Feature Renaming: shorten long column names; skip columns not present
rename_map = {
    "start_datetime": "start",
    "missing_links": "mis_link",
    "radio_silence": "silence",
    "code_only_devs": "code_dev",
    "code_files": "file",
    "ml_only_devs": "mail_dev",
    "ml_threads": "thread",
    "n_commits": "commit"
}
raw_dt = raw_dt.rename(columns={k: v for k, v in rename_map.items() if k in raw_dt.columns})

# Fill any required columns missing from this dataset with 0
if "org_silo" not in raw_dt.columns:
    raw_dt["org_silo"] = 0

required_cols = [
    "cve_id", "activity_0", "activity_2", "start",
    "org_silo", "mis_link", "silence", "code_dev", "file",
    "mail_dev", "thread", "commit", "churn"
]
for col in required_cols:
    if col not in raw_dt.columns:
        raw_dt[col] = 0

dt = raw_dt[required_cols].copy()
print(f"After Feature Renaming: {dt.shape[0]} rows × {dt.shape[1]} columns")

After Feature Renaming: 4870 rows × 13 columns


/var/folders/qg/19_8zd9x78b4fcgq8j_d8jmc0000gn/T/ipykernel_12215/1293814442.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw_dt["org_silo"] = 0


## Missing Data Handling

We decided to remove rows from the dataset for which the mailing list data source is missing (i.e. 2000-2001).

In [7]:
# Convert start to datetime for year-based filtering, handling both
# numeric Unix timestamps and string datetime formats
if pd.api.types.is_numeric_dtype(dt["start"]):
    start_dt_series = pd.to_datetime(dt["start"], unit='s', utc=True, errors='coerce')
else:
    start_dt_series = pd.to_datetime(dt["start"], errors="coerce", utc=True)

mask = (start_dt_series.dt.year < 2000) | (start_dt_series.dt.year > 2001)
dt = dt[mask].copy()
start_dt_series = start_dt_series[mask]

dt["start"] = start_dt_series.map(lambda x: x.timestamp() if pd.notna(x) else np.nan)
dt = dt.fillna(0)
print(f"After Missing Data Transformations: {dt.shape[0]} rows × {dt.shape[1]} columns")

After Missing Data Transformations: 4870 rows × 13 columns


## 1-Time Lag Features

In [8]:
lag_feature_cols = [
    "org_silo", "mis_link", "silence", "code_dev", "file",
    "mail_dev", "thread", "commit", "churn"
]

dt = dt.sort_values(["cve_id", "start"]).reset_index(drop=True)

def add_time_lag(cve_table: pd.DataFrame) -> pd.DataFrame:
    cve_table = cve_table.copy()
    if len(cve_table) < 2:
        for col in lag_feature_cols:
            cve_table[f"{col}2"] = np.nan
        return cve_table

    for col in lag_feature_cols:
        cve_table[f"{col}2"] = cve_table[col].shift(-1)
    return cve_table.iloc[:-1].copy()

lag_parts = [add_time_lag(group) for _, group in dt.groupby("cve_id", sort=False)]
lag_dt = pd.concat(lag_parts, ignore_index=True)
print(f"After Appending Next Time Period Variables: {lag_dt.shape[0]} rows × {lag_dt.shape[1]} columns")

After Appending Next Time Period Variables: 4772 rows × 22 columns


## Remove Short CVEs

We deleted CVEs (their associated rows) with 7 or fewer time periods.

In [9]:
cve_counts = lag_dt.groupby("cve_id").size()
short_cve_ids = cve_counts[cve_counts <= 7].index
lag_dt = lag_dt[~lag_dt["cve_id"].isin(short_cve_ids)].copy()
print(f"After Removing Short CVEs: {lag_dt.shape[0]} rows × {lag_dt.shape[1]} columns")

After Removing Short CVEs: 4772 rows × 22 columns


## Addressing Determinism and High Intercorrelation Among Features

Due to high correlation, we perform 6 feature deletions (activity_0, activity_2, org_silo, org_silo2):

In [10]:
selected_cols = [
    "cve_id", "start",
    "mis_link", "silence", "code_dev", "file",
    "mail_dev", "thread", "commit", "churn",
    "mis_link2", "silence2", "code_dev2", "file2",
    "mail_dev2", "thread2", "commit2", "churn2"
]
lag_dt = lag_dt[selected_cols].copy()
print(f"After Correlation/Determinism Pruning: {lag_dt.shape[0]} rows × {lag_dt.shape[1]} columns")

After Correlation/Determinism Pruning: 4772 rows × 18 columns


## Binarized CVE Indicators

To represent the CVE Ids, we utilize indicator features. For every CVE ID, a new column is added to the table which can take values 0 or 1. The value is 1 if the row is associated to that CVE ID, or 0 otherwise.

We can then remove the `cve_id` column, as the binary features represent the same information, and add the remaining columns to the analysis table:

In [11]:
cve_binarized = pd.get_dummies(
    lag_dt["cve_id"].astype("Int64").astype(str),
    prefix="b",
    dtype=int
)

binarized_lag_dt = pd.concat([
    lag_dt.drop(columns=["cve_id"]).reset_index(drop=True),
    cve_binarized.reset_index(drop=True)
], axis=1)
print(f"After Binarize CVE ID: {binarized_lag_dt.shape[0]} rows × {binarized_lag_dt.shape[1]} columns")

After Binarize CVE ID: 4772 rows × 115 columns


## Add Null Features

## Keep only 5 null indicator features

Introducing a null feature for all variables and features leads to too many features being introduced for causal search, causing heap memory errors in Tetrad. We preserve only a few of the nv binary indicator variables, as they lead to variable explosion and their pattern is easy to randomize. Position 138 includes all variables as null variables, plus five binary indicators as null variables. We consider this loss of null binary indicator features reasonable, as the randomization of a few blocks of values 1 or 0 will generally be equivalent. This in turn, allow us to perform more causal search runs, which we deem a fair trade-off. 

In [12]:
rng = np.random.default_rng(32)
null_df = pd.DataFrame(
    {col: rng.permutation(binarized_lag_dt[col].to_numpy()) for col in binarized_lag_dt.columns}
)
null_df.columns = [f"nv-{col}" for col in null_df.columns]

null_non_indicator_cols = [c for c in null_df.columns if not c.startswith("nv-b_")]
null_indicator_cols = [c for c in null_df.columns if c.startswith("nv-b_")]
null_keep_cols = null_non_indicator_cols + null_indicator_cols[:5]
null_df = null_df[null_keep_cols]

processed_dt = pd.concat([binarized_lag_dt, null_df], axis=1)

# Save outputs to separate files — input DATA_PATH is never modified
processed_dt.to_csv(PROCESSED_DATA_PATH, index=False)
binarized_lag_dt.to_csv(BINARIZED_DATA_PATH, index=False)

print(f"✓ Saved null-variable dataset : {PROCESSED_DATA_PATH} ({processed_dt.shape[0]} × {processed_dt.shape[1]})")
print(f"✓ Saved non-null dataset      : {BINARIZED_DATA_PATH} ({binarized_lag_dt.shape[0]} × {binarized_lag_dt.shape[1]})")

# Use processed data in-memory for the rest of the notebook
data = processed_dt.copy()
data = data.astype({col: "float64" for col in data.columns})
print(f"Dataset: {data.shape[0]} rows × {data.shape[1]} columns")

✓ Saved null-variable dataset : null_variable_processed_dt.csv (4772 × 137)
✓ Saved non-null dataset      : binarized_variable_dt.csv (4772 × 115)
Dataset: 4772 rows × 137 columns


## Variable Analysis

Identify binary CVE indicators, continuous metrics, and null variables.

In [13]:
b_cols      = [col for col in data.columns if col.startswith('b_')]
nv_cols     = [col for col in data.columns if col.startswith('nv-')]
metric_cols = [col for col in data.columns if not col.startswith('b_') and not col.startswith('nv-')]

print(f"Binary indicators: {len(b_cols)}")
print(f"Continuous metrics: {len(metric_cols)}")
print(f"Null variables: {len(nv_cols)}")

Binary indicators: 98
Continuous metrics: 17
Null variables: 22


# FGES Null Variable Search

Executes causal discovery with bootstrapping over the null-variable dataset.

In [14]:
# Matches R: data_io + algorithm_boss/fges + score_sem_bic + bootstrapping + tetrad()
# Note: No domain knowledge constraints in null variable search (see R notebook)
search = ts.TetradSearch(data)
search.use_sem_bic(penalty_discount=2, sem_bic_rule=1,
                   structurePrior=0, singularity_lambda=0.0)

if ALGORITHM == "boss":
    # BOSS: 100% resample size (R: percent_resample_size=100, number_resampling=1000)
    search.set_bootstrapping(numberResampling=N_BOOTSTRAP, percent_resample_size=100,
                             seed=32, add_original=True, with_replacement=True,
                             resampling_ensemble=1)
    search.set_verbose(verbose=False)
    start_time = time.time()
    search.run_boss(num_starts=1, use_bes=False, time_lag=0,
                    use_data_order=True, output_cpdag=True)
else:
    # FGES: 90% resample size (R: percent_resample_size=90, number_resampling=500)
    search.set_bootstrapping(numberResampling=N_BOOTSTRAP, percent_resample_size=90,
                             seed=32, add_original=True, with_replacement=True,
                             resampling_ensemble=1)
    search.set_verbose(verbose=False)
    start_time = time.time()
    search.run_fges(max_degree=1000, faithfulness_assumed=True,
                    symmetric_first_step=True, parallelized=False)

elapsed = time.time() - start_time
graph   = search.get_java()
print(f"Discovered: {graph.getNumNodes()} nodes, {graph.getNumEdges()} edges")
print(f"Elapsed: {elapsed:.1f}s")

null_graph_json = str(search.get_json())

Bootstrap count = 1
Bootstrap count = 2
Bootstrap count = 3
Bootstrap count = 4
Bootstrap count = 5
Bootstrap count = 6
Bootstrap count = 7
Bootstrap count = 8
Bootstrap count = 9
Bootstrap count = 10
Bootstrap count = 11
Bootstrap count = 12
Bootstrap count = 13
Bootstrap count = 14
Bootstrap count = 15
Bootstrap count = 16
Bootstrap count = 17
Bootstrap count = 18
Bootstrap count = 19
Bootstrap count = 20
Bootstrap count = 21
Bootstrap count = 22
Bootstrap count = 23
Bootstrap count = 24
Bootstrap count = 25
Bootstrap count = 26
Bootstrap count = 27
Bootstrap count = 28
Bootstrap count = 29
Bootstrap count = 30
Bootstrap count = 31
Bootstrap count = 32
Bootstrap count = 33
Bootstrap count = 34
Bootstrap count = 35
Bootstrap count = 36
Bootstrap count = 37
Bootstrap count = 38
Bootstrap count = 39
Bootstrap count = 40
Bootstrap count = 41
Bootstrap count = 42
Bootstrap count = 43
Bootstrap count = 44
Bootstrap count = 45
Bootstrap count = 46
Bootstrap count = 47
Bootstrap count = 48
B

---

# Deriving the 1 PNEF Threshold

In our causal search above, we introduced null features over multiple bootstrap runs to observe how often our causal search forms random edges (i.e. between our features and null features). We will use this information to derive a threshold, **1PNEF** (1st Percentile NoEdge Frequency), we can use in our final causal search.

## Graph Examination

We parse the Tetrad JSON graph output into tabular format: nodes, edgeset, and edge type probabilities.

The **edgeset** table contains the ensemble edge for each node pair. Because we performed multiple bootstrap runs, the probabilities represent the ensemble of all edges formed on each execution.

The **edge_type_probabilities** table shows the counts of each type of edge formed on each subgraph across all bootstrap runs.

In [15]:
null_graph = parse_graph(null_graph_json)

print(f"Nodes: {len(null_graph['nodes'])}")
print(f"\nFirst 5 nodes:")
print(null_graph['nodes'].head())
print(f"\nEdgeset: {len(null_graph['edgeset'])} edges")
print(null_graph['edgeset'].head())
print(f"\nEdge type probabilities: {len(null_graph['edge_type_probabilities'])} entries")
print(null_graph['edge_type_probabilities'].head())

Nodes: 137

First 5 nodes:
  node_name
0  b_100433
1  b_100740
2  b_100742
3  b_102939
4  b_103864

Edgeset: 96 edges
  node1_name node2_name endpoint1 endpoint2   bold  highlighted properties  \
0  code_dev2   code_dev      TAIL     ARROW  False        False      dd;nl   
1      start   b_160705      TAIL     ARROW  False        False      pd;nl   
2    silence   silence2      TAIL     ARROW  False        False      dd;nl   
3   mail_dev    silence      TAIL     ARROW  False        False      dd;pl   
4      start    commit2      TAIL     ARROW  False        False      dd;nl   

   probability  
0     1.000000  
1     0.691358  
2     0.839506  
3     1.000000  
4     0.864198  

Edge type probabilities: 292 entries
  node1_name node2_name edge_type properties  probability
0  code_dev2   code_dev        ta      dd;nl     0.506173
1  code_dev2   code_dev        at      pd;nl     0.481481
2  code_dev2   code_dev        tt        NaN     0.012346
3      start   b_160705        ta      pd

## Deriving 1 PNEF

Our interest is to derive a threshold for the final causal search, using the information from this bootstrapped null feature causal search. By definition, edges formed between actual variables and random (null) features represent random edges.

We:
1. Subset the edgeset to contain only edges where at least one node is a null variable (nv-*)
2. Derive a `no_edge` probability by subtracting the probability from 1
3. Identify the 1st percentile value of the no_edge probability → the **1PNEF threshold**

This threshold tells us: given entirely random variables, causal links were formed between them up to X% of the time. In our final search, we only keep causal links that formed **more** than X% of the time.

In [16]:
nv_edges = null_graph['edgeset'].copy()
is_node1_nv = nv_edges['node1_name'].str.contains('nv-', regex=False)
is_node2_nv = nv_edges['node2_name'].str.contains('nv-', regex=False)
nv_edges = nv_edges[is_node1_nv | is_node2_nv]
nv_edges.head()

nv_edges['no_edge'] = 1 - nv_edges['probability']

pnef_1 = float(nv_edges['no_edge'].quantile(0.01))
pnef_1

0.3827160493827161

---

# Non-Null Causal Search

With the threshold defined, we now proceed to the final causal search, which **does not include null features**. In this non-null feature causal search, we also specify domain knowledge to prohibit causal links that don't make sense temporally (e.g. features at 1-time-lag cannot cause features in the present).

## Domain Knowledge Causal Search without Null Variables

Remove null variable columns (nv-*) from the dataset, keeping only the original features and binary CVE indicators.

In [17]:
nv_cols       = [col for col in data.columns if col.startswith('nv-')]
non_null_data = data.drop(columns=nv_cols)
non_null_data.to_csv(BINARIZED_DATA_PATH, index=False)
print(f"Non-null dataset: {non_null_data.shape[0]} rows × {non_null_data.shape[1]} columns")
print(f"Saved to: {BINARIZED_DATA_PATH}")

Non-null dataset: 4772 rows × 115 columns
Saved to: binarized_variable_dt.csv


## Causal Search

Run the causal search on the non-null dataset with domain knowledge constraints. This search uses the same algorithm and bootstrap settings, but on the dataset **without** null features and **with** temporal knowledge constraints.

In [18]:
# Matches R: data_io + knowledge_flags + algorithm_boss/fges + score_sem_bic + bootstrapping + tetrad()
# Domain knowledge prohibits lag-2 features from causing lag-1 features (temporal ordering)
domain_search = ts.TetradSearch(non_null_data)
domain_search.use_sem_bic(penalty_discount=2, sem_bic_rule=1,
                           structurePrior=0, singularity_lambda=0.0)

if ALGORITHM == "boss":
    domain_search.set_bootstrapping(numberResampling=N_BOOTSTRAP, percent_resample_size=100,
                                     seed=32, add_original=True, with_replacement=True,
                                     resampling_ensemble=1)
    domain_search.set_verbose(verbose=False)
    if os.path.exists(KNOWLEDGE_FILE):
        domain_search.load_knowledge(KNOWLEDGE_FILE)
        print(f"Knowledge loaded from: {KNOWLEDGE_FILE}")
    start_time = time.time()
    domain_search.run_boss(num_starts=1, use_bes=False, time_lag=0,
                            use_data_order=True, output_cpdag=True)
else:
    domain_search.set_bootstrapping(numberResampling=N_BOOTSTRAP, percent_resample_size=90,
                                     seed=32, add_original=True, with_replacement=True,
                                     resampling_ensemble=1)
    domain_search.set_verbose(verbose=False)
    if os.path.exists(KNOWLEDGE_FILE):
        domain_search.load_knowledge(KNOWLEDGE_FILE)
        print(f"Knowledge loaded from: {KNOWLEDGE_FILE}")
    start_time = time.time()
    domain_search.run_fges(max_degree=1000, faithfulness_assumed=True,
                            symmetric_first_step=True, parallelized=False)

elapsed          = time.time() - start_time
domain_graph_obj = domain_search.get_java()
print(f"Discovered: {domain_graph_obj.getNumNodes()} nodes, {domain_graph_obj.getNumEdges()} edges")
print(f"Elapsed: {elapsed:.1f}s")

domain_graph_json = str(domain_search.get_json())


Loading knowledge.
Adding to tier 0 anti_motif_square
Adding to tier 0 anti_triangle_motif
Adding to tier 0 clique
Adding to tier 0 crossing
Adding to tier 0 modularity-violation
Adding to tier 0 package-cycle
Adding to tier 0 unhealthy-inheritance
Adding to tier 0 unstable-interface
Adding to tier 1 file_bug_churn
Adding to tier 1 file_bug_frequency
Adding to tier 1 file_churn
Adding to tier 1 file_non_bug_churn
Adding to tier 1 file_non_bug_frequency
Knowledge loaded from: mike_knowledge_box.txt
Bootstrap count = 1
Bootstrap count = 2
Bootstrap count = 3
Bootstrap count = 4
Bootstrap count = 5
Bootstrap count = 6
Bootstrap count = 7
Bootstrap count = 8
Bootstrap count = 9
Bootstrap count = 10
Bootstrap count = 11
Bootstrap count = 12
Bootstrap count = 13
Bootstrap count = 14
Bootstrap count = 15
Bootstrap count = 16
Bootstrap count = 17
Bootstrap count = 18
Bootstrap count = 19
Bootstrap count = 20
Bootstrap count = 21
Bootstrap count = 22
Bootstrap count = 23
Bootstrap count = 24
B

## Graph Examination

Parse the domain knowledge causal search JSON output into nodes, edgeset, and edge type probabilities.

In [19]:
# Parse the domain search JSON output
domain_graph = parse_graph(domain_graph_json)

print(f"Domain search nodes: {len(domain_graph['nodes'])}")
print(f"Domain search edges: {len(domain_graph['edgeset'])}")
print(f"\nEdgeset sample:")
domain_graph['edgeset'].head()


Domain search nodes: 115
Domain search edges: 96

Edgeset sample:


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability
0,code_dev2,code_dev,TAIL,ARROW,False,False,dd;nl,1.000000
1,start,b_160705,TAIL,ARROW,False,False,pd;nl,0.666667
2,mail_dev,silence,TAIL,ARROW,False,False,dd;pl,1.000000
3,silence,silence2,TAIL,ARROW,False,False,dd;nl,0.839506
4,churn,commit2,TAIL,ARROW,False,False,dd;nl,0.629630


---

# Applying 1PNEF Threshold

## Applying 1PNEF Threshold

Edges which may have been formed at random are filtered here. We apply the 1PNEF threshold derived from the null variable search to the domain knowledge search results. Only edges whose `no_edge` probability is less than or equal to the 1PNEF threshold are kept.

In [20]:
# Applying 1PNEF Threshold (R-equivalent explicit steps)
edges = domain_graph['edgeset'].copy()
edges['no_edge'] = 1 - edges['probability']
edges_1pnef = edges[edges['no_edge'] <= pnef_1].copy()

print(f"\nFiltered edges (1PNEF trimmed): {len(edges_1pnef)}")
edges_1pnef.head(20)


Filtered edges (1PNEF trimmed): 90


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability,no_edge
0,code_dev2,code_dev,TAIL,ARROW,False,False,dd;nl,1.000000,0.000000
1,start,b_160705,TAIL,ARROW,False,False,pd;nl,0.666667,0.333333
2,mail_dev,silence,TAIL,ARROW,False,False,dd;pl,1.000000,0.000000
3,silence,silence2,TAIL,ARROW,False,False,dd;nl,0.839506,0.160494
4,churn,commit2,TAIL,ARROW,False,False,dd;nl,0.629630,0.370370
5,start,commit2,TAIL,ARROW,False,False,dd;nl,0.876543,0.123457
6,thread2,churn2,TAIL,ARROW,False,False,pd;nl,0.962963,0.037037
7,mis_link,silence2,TAIL,ARROW,False,False,dd;nl,0.790123,0.209877
8,mis_link2,silence2,TAIL,ARROW,False,False,dd;pl,0.827160,0.172840
9,file2,thread2,TAIL,ARROW,False,False,NaN,0.641975,0.358025


In [21]:
# Save the 1PNEF-filtered edges
os.makedirs(NON_NULL_OUTPUT_DIR, exist_ok=True)
edges_1pnef.to_csv(f"{NON_NULL_OUTPUT_DIR}/edges_1pnef.csv", index=False)
print(f"✓ Saved filtered edges to {NON_NULL_OUTPUT_DIR}/edges_1pnef.csv")

✓ Saved filtered edges to boss_domain_results/edges_1pnef.csv


---

# Results

With the final causal graph trimmed, we can now inspect it to draw conclusions. Causal graphs may form cycles and have undirected edges.

## Full Causal Graph 1-PNEF Trimmed

Interactive visualization of the full causal graph after applying the 1PNEF threshold.

Edge colors:
- **Black**: Directed edges (causal relationship)
- **Red**: Undirected edges (TAIL-TAIL, association without determined direction)

In [22]:
# try:
#     from pyvis.network import Network
#     _pyvis_available = True
# except ImportError:
#     print("⚠️  pyvis not installed. Install with: pip install pyvis")
#     _pyvis_available = False
#
# # Color scheme matching R colorBlindness::Blue2DarkRed12Steps
# _COLORBLIND_PALETTE = {
#     'b_':       '#2166AC',  # Blue — binary indicators
#     'mis_link': '#67A9CF',  # Light blue
#     'silence':  '#D1E5F0',  # Pale blue
#     'code_dev': '#FDDBC7',  # Pale orange
#     'churn':    '#F4A582',  # Light red
#     'commit':   '#D6604D',  # Red
#     'default':  '#67A9CF',  # Default: light blue
# }
#
# def _node_color(name):
#     for key, color in _COLORBLIND_PALETTE.items():
#         if key != 'default' and key in name:
#             return color
#     return _COLORBLIND_PALETTE['default']
#
# # Prepare edges for visualization
# viz_edges = edges_1pnef.copy()
# viz_edges['color'] = 'black'
# viz_edges.loc[(viz_edges['endpoint1'] == 'TAIL') & (viz_edges['endpoint2'] == 'TAIL'), 'color'] = 'red'
# viz_edges = viz_edges.rename(columns={'node1_name': 'from', 'node2_name': 'to'})
# viz_edges['weight'] = viz_edges['probability']
# viz_edges['label']  = viz_edges['probability'].round(3).astype(str)
#
# if _pyvis_available and len(viz_edges) > 0:
#     net = Network(height="700px", width="100%", directed=True,
#                   notebook=True, cdn_resources='in_line')
#
#     all_node_names = set(viz_edges['from'].tolist() + viz_edges['to'].tolist())
#     all_node_names.update(domain_graph['nodes']['node_name'].tolist())
#     for n in all_node_names:
#         net.add_node(n, label=n, color=_node_color(n), title=n, size=20)
#
#     for _, row in viz_edges.iterrows():
#         arrows = 'to' if row.get('endpoint2') == 'ARROW' else (
#                  'from' if row.get('endpoint1') == 'ARROW' else '')
#         net.add_edge(row['from'], row['to'],
#                      color=row['color'],
#                      value=float(row['weight']),
#                      title=f"p={row['weight']}",
#                      label=row['label'],
#                      arrows=arrows)
#
#     net.set_options("""
#     {
#       "physics": {
#         "forceAtlas2Based": {
#           "gravitationalConstant": -50,
#           "centralGravity": 0.01,
#           "springLength": 200,
#           "springConstant": 0.08
#         },
#         "solver": "forceAtlas2Based",
#         "stabilization": {"iterations": 150}
#       },
#       "interaction": {"navigationButtons": true, "keyboard": true, "hover": true}
#     }
#     """)
#
#     output_html = f"{NON_NULL_OUTPUT_DIR}/causal_graph_full.html"
#     os.makedirs(NON_NULL_OUTPUT_DIR, exist_ok=True)
#     net.save_graph(output_html)
#     print(f"Graph saved to: {output_html}")
#     net.show(output_html)
# else:
#     print(f"\nFull causal graph: {len(viz_edges)} edges")
#     print(viz_edges[['from', 'to', 'probability', 'color']].to_string(index=False))

<!-- ## Sub-Graphs of Effort Variables and Parents

Focus on key effort variables and their neighboring causal structure in a smaller sub-graph. -->

In [23]:
# if _pyvis_available and len(viz_edges) > 0:
#     # Include edges between nodes of interest AND edges from parents into nodes of interest
#     sub_mask = (
#         (viz_edges['from'].isin(NODES_OF_INTEREST) & viz_edges['to'].isin(NODES_OF_INTEREST)) |
#         viz_edges['to'].isin(NODES_OF_INTEREST)
#     )
#     sub_edges = viz_edges[sub_mask].copy()
#
#     if len(sub_edges) == 0:
#         print("No edges found between specified nodes of interest.")
#     else:
#         sub_node_names = set(sub_edges['from'].tolist() + sub_edges['to'].tolist())
#         print(f"Sub-graph: {len(sub_node_names)} nodes, {len(sub_edges)} edges")
#
#         sub_net = Network(height="700px", width="100%", directed=True,
#                           notebook=True, cdn_resources='in_line')
#
#         for n in sub_node_names:
#             sub_net.add_node(n, label=n, color=_node_color(n), title=n, size=20)
#
#         for _, row in sub_edges.iterrows():
#             arrows = 'to' if row.get('endpoint2') == 'ARROW' else (
#                      'from' if row.get('endpoint1') == 'ARROW' else '')
#             sub_net.add_edge(row['from'], row['to'],
#                              color=row['color'],
#                              value=float(row['weight']),
#                              title=f"p={row['weight']}",
#                              label=row['label'],
#                              arrows=arrows)
#
#         sub_net.set_options("""
#         {
#           "physics": {
#             "forceAtlas2Based": {
#               "gravitationalConstant": -50,
#               "centralGravity": 0.01,
#               "springLength": 200,
#               "springConstant": 0.08
#             },
#             "solver": "forceAtlas2Based",
#             "stabilization": {"iterations": 150}
#           },
#           "interaction": {"navigationButtons": true, "keyboard": true, "hover": true}
#         }
#         """)
#
#         sub_output_html = f"{NON_NULL_OUTPUT_DIR}/causal_graph_subgraph.html"
#         os.makedirs(NON_NULL_OUTPUT_DIR, exist_ok=True)
#         sub_net.save_graph(sub_output_html)
#         print(f"Sub-graph saved to: {sub_output_html}")
#         sub_net.show(sub_output_html)
# else:
#     print("No edges to display in sub-graph.")

<!-- ## Cycle Detection

Check if the causal graph contains any cycles. Cycles indicate feedback loops in the causal structure. -->

In [24]:
# try:
#     import networkx as nx
#     _nx_available = True
# except ImportError:
#     print("⚠️  networkx not installed. Install with: pip install networkx")
#     _nx_available = False
#
# if _nx_available and len(viz_edges) > 0:
#     # Build directed graph from directed (non-TAIL-TAIL) edges only
#     G = nx.DiGraph()
#     directed_edges = viz_edges[viz_edges['color'] == 'black']
#     for _, row in directed_edges.iterrows():
#         G.add_edge(row['from'], row['to'])
#
#     cycles = list(nx.simple_cycles(G))
#
#     if not cycles:
#         print("No cycles detected in the graph.")
#     else:
#         print(f"Found {len(cycles)} cycle(s):")
#         for i, cycle in enumerate(cycles, 1):
#             cycle_str = " → ".join(cycle) + f" → {cycle[0]}"
#             print(f"  Cycle {i} (length {len(cycle)}): {cycle_str}")
# else:
#     print("Cycle detection skipped (networkx unavailable or no edges).")